# Bridge Vibration Simulation — Notebook Walkthrough

This notebook drives the `bridge_sim` package end-to-end:
1. Modal analysis (natural frequencies & mode shapes)
2. Heavy truck crossing
3. Earthquake (synthetic Kanai-Tajimi)
4. Turbulent wind
5. Mixed traffic
6. Damage detection by frequency shift
7. **Interactive viewer** with scenario buttons (last cell)

## Before running
Make sure the project root (the folder containing `bridge_sim/`, `examples/`, `environment.yml`) is your Jupyter working directory, then run the cells in order.

## 0 · Backend setup

Static plots use the default inline backend. The **last cell** needs an *interactive* backend so buttons and animation actually work:

| Backend | Command | When to use |
|---|---|---|
| `widget`  | `%matplotlib widget`  | JupyterLab / Jupyter Notebook 7 — needs `ipympl` (`conda install -c conda-forge ipympl`) |
| `notebook`| `%matplotlib notebook`| Classic Jupyter Notebook only |
| `qt`      | `%matplotlib qt`      | Pops a separate window — needs `pyqt` (`conda install pyqt`) |
| `tk`      | `%matplotlib tk`      | Pops a separate window — usually preinstalled |

For now we stay inline; the interactive backend is switched on right before the viewer cell.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Make sure the project root is on sys.path so `bridge_sim` is importable
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from bridge_sim import (
    Bridge, modal_analysis,
    HEAVY_TRUCK, CAR, VAN, BUS, TRUCK, Vehicle,
    moving_load, traffic_stream,
    Earthquake, WindBuffeting,
    newmark_beta, frequency_response,
)
from bridge_sim.damage import damage_sweep
from bridge_sim.post import (
    plot_mode_shapes, plot_bridge_schematic_with_modes,
    plot_time_history, plot_spectrum, plot_damage_sweep,
    midspan_deflection, fft_spectrum,
)

print('Package imported OK')

## 1 · Modal analysis

Default bridge: 60 m simply-supported steel box girder (8.0 × 2.5 m, t = 40 mm), 40 finite elements, 80 free DOFs. We solve the generalised eigenproblem $(K - \omega^2 M)\,\varphi = 0$ on the constrained system.

In [ ]:
bridge = Bridge(length=60.0, n_elements=40)
print(bridge.summary())

modes = modal_analysis(bridge, n_modes=6)
print('\nNatural frequencies (Hz):')
for k, f in enumerate(modes.natural_frequencies_hz, start=1):
    print(f'  Mode {k}: {f:7.3f} Hz')

# Sanity check vs Euler-Bernoulli: f_n / f_1 should be 1, 4, 9, 16, 25, 36
ratios = modes.natural_frequencies_hz / modes.natural_frequencies_hz[0]
print('\nRatios f_n / f_1   (theory: 1, 4, 9, 16, 25, 36):')
print(' ', np.round(ratios, 3))

In [ ]:
plot_mode_shapes(bridge, modes, 'figures/nb_01_modes.png', n_modes=3)
from IPython.display import Image
Image('figures/nb_01_modes.png')

## 2 · Heavy truck crossing
A 44-tonne articulated lorry crosses at 50 km/h. The moving point load is distributed onto the FE nodes via Hermite shape functions; the system is integrated with Newmark-β (γ=½, β=¼).

In [ ]:
load = moving_load(bridge, HEAVY_TRUCK)
cross_t  = bridge.length / HEAVY_TRUCK.speed_mps
duration = cross_t + 5.0

th = newmark_beta(bridge, load, duration_s=duration, dt=0.005,
                  zeta_1=0.02, zeta_2=0.02)
w_mid = midspan_deflection(bridge, th) * 1000   # mm

print(f'Crossing time      : {cross_t:.2f} s')
print(f'Peak mid-span sag  : {np.max(np.abs(w_mid)):.2f} mm')

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(th.t, w_mid, color='#1E2761')
ax.set(xlabel='Temps t [s]', ylabel='Déplacement mi-travée [mm]',
       title="Poids-lourd 44 t à 50 km/h — déflexion mi-travée")
ax.grid(alpha=.3); plt.show()

In [ ]:
# Spectrum of the free-vibration tail
f, A = fft_spectrum(th.t, w_mid)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.fill_between(f, 0, A, color='#5BA3D0', alpha=0.5); ax.plot(f, A, color='#1E2761')
for fn in modes.natural_frequencies_hz[:4]:
    ax.axvline(fn, color='#C0392B', lw=0.9, ls=':')
    ax.text(fn, A.max()*0.9, f' {fn:.2f} Hz', color='#C0392B', rotation=90, fontsize=8, va='top')
ax.set(xlim=(0, 20), xlabel='Fréquence f [Hz]', ylabel='Amplitude',
       title='Spectre — passage poids-lourd')
ax.grid(alpha=.3); plt.show()

## 3 · Earthquake (Kanai-Tajimi)
Synthetic ground motion with PGA = 0.30 g, dominant 2.5 Hz (close to f₁ ⇒ near-resonance). The equivalent load is $F_\mathrm{eq}(t) = -M\,r\,a_g(t)$.

In [ ]:
eq = Earthquake(bridge, duration_s=20.0, dt=0.01, pga_g=0.30,
                dominant_frequency_hz=2.5)
th_eq = newmark_beta(bridge, eq, duration_s=20.0, dt=0.005,
                     zeta_1=0.02, zeta_2=0.02)
w_eq = midspan_deflection(bridge, th_eq) * 1000

fig, axes = plt.subplots(2, 1, figsize=(9, 4.8), sharex=True)
axes[0].plot(eq.t_grid, eq.a_g/9.81, color='#C0392B')
axes[0].set(ylabel='a_g [g]', title=f'PGA = 0.30 g · f_dom = 2.5 Hz')
axes[0].grid(alpha=.3)
axes[1].plot(th_eq.t, w_eq, color='#1E2761')
axes[1].set(xlabel='t [s]', ylabel='Mid-span [mm]', title='Bridge response')
axes[1].grid(alpha=.3); plt.tight_layout(); plt.show()

print(f'Peak mid-span deflection under earthquake: {np.max(np.abs(w_eq)):.2f} mm')

## 4 · Turbulent wind

In [ ]:
wind = WindBuffeting(bridge, mean_speed_mps=25.0, turbulence_intensity=0.12,
                     duration_s=20.0, dt_sample=0.01)
th_w = newmark_beta(bridge, wind, duration_s=20.0, dt=0.005,
                    zeta_1=0.02, zeta_2=0.02)
w_w = midspan_deflection(bridge, th_w) * 1000
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(th_w.t, w_w, color='#1E8449'); ax.grid(alpha=.3)
ax.set(xlabel='t [s]', ylabel='Mid-span [mm]',
       title='Vent turbulent V̄ = 25 m/s, I_u = 12 %')
plt.show()
print(f'Peak (small): {np.max(np.abs(w_w)):.3f} mm  ·  std: {np.std(w_w):.3f} mm')

## 5 · Mixed traffic

In [ ]:
fleet = [
    Vehicle(CAR.mass_kg,         CAR.speed_mps,         entry_time_s=0.0,  label='car 1'),
    Vehicle(VAN.mass_kg,         VAN.speed_mps,         entry_time_s=0.6,  label='van'),
    Vehicle(BUS.mass_kg,         BUS.speed_mps,         entry_time_s=1.4,  label='bus'),
    Vehicle(TRUCK.mass_kg,       TRUCK.speed_mps,       entry_time_s=2.6,  label='truck'),
    Vehicle(HEAVY_TRUCK.mass_kg, HEAVY_TRUCK.speed_mps, entry_time_s=4.0,  label='44-t lorry'),
    Vehicle(CAR.mass_kg,         CAR.speed_mps,         entry_time_s=4.6,  label='car 2'),
]
th_t = newmark_beta(bridge, traffic_stream(bridge, fleet),
                    duration_s=12.0, dt=0.005, zeta_1=0.02, zeta_2=0.02)
w_t = midspan_deflection(bridge, th_t) * 1000
plt.figure(figsize=(9, 3.2))
plt.plot(th_t.t, w_t, color='#1E2761'); plt.grid(alpha=.3)
plt.xlabel('t [s]'); plt.ylabel('Mid-span [mm]')
plt.title('Mixed traffic — 6 vehicles staggered')
plt.show()
print(f'Peak under traffic: {np.max(np.abs(w_t)):.2f} mm')

## 6 · Damage detection — frequency sensitivity

We progressively reduce the stiffness of a mid-span element and track f₁…f₄. Modes whose deformed shape has an *antinode* at mid-span (modes 1, 3) are sensitive; modes with a *node* there (modes 2, 4) are not.

In [ ]:
mid_el = bridge.n_elements // 2
losses = np.linspace(0.0, 0.6, 13)
losses, freqs = damage_sweep(bridge, mid_el, losses, n_modes=4)

plt.figure(figsize=(9, 3.6))
for k, col in enumerate(['#1E2761', '#2A6DB0', '#5BA3D0', '#E67E22']):
    plt.plot(losses*100, freqs[:, k]/freqs[0, k]*100,
             marker='o', color=col, label=f'Mode {k+1}')
plt.xlabel('Stiffness loss in mid-span element [%]')
plt.ylabel('f / f_intact [%]')
plt.legend(); plt.grid(alpha=.3); plt.title('Damage signature')
plt.show()

## 7 · Interactive viewer

Now we launch the live viewer. Switching the backend below is essential — without it the buttons and animation won't react.

Pick the backend that matches your setup:

In [ ]:
# JupyterLab / Notebook 7 with ipympl installed (recommended):
%matplotlib widget

# --- Alternatives — uncomment ONE if the above doesn't work ---
# %matplotlib notebook   # classic Jupyter Notebook only
# %matplotlib qt         # opens a separate window (needs pyqt)
# %matplotlib tk         # opens a separate window (usually preinstalled)

from examples.visual_simulation import BridgeViewer
viewer = BridgeViewer()
viewer.fig   # display the figure as the cell output

Click any scenario button on the right. The first click triggers the Newmark-β computation (a few seconds); subsequent clicks replay from cache. Use **Pause / Play**, **Restart**, and **← slow / fast →** to control playback.

Tip: if `%matplotlib widget` says it can't find `ipympl`, install it from a terminal with the environment active:
```bash
conda install -c conda-forge ipympl
```
then restart the kernel and re-run the cell.